In [1]:
import os
import requests
import json
from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_experimental.sql import SQLDatabaseSequentialChain
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from typing_extensions import Annotated, TypedDict
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
# from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor
from langchain_openai import ChatOpenAI
import psycopg2
import time 
import yaml
import json 
from langchain_community.agent_toolkits import JsonToolkit, create_json_agent
from langchain_community.tools.json.tool import JsonSpec
from langchain_openai import OpenAI
from langchain.tools import Tool

### Add Thingsboard details

In [7]:
THINGSBOARD_URL = "http://localhost:8080"
USERNAME = "tenant@thingsboard.org"
PASSWORD = "tenant"
DASHBOARD_ID = "http://localhost:8080/tenants"

DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "thingsboard"
DB_USER = "thingsboard"
DB_PASSWORD = "postgres"


THINGSBOARD_HOST = "http://localhost:8080"
USERNAME = "tenant@thingsboard.org"
PASSWORD = "tenant"

# Authenticate and get the JWT token
auth_url = f'{THINGSBOARD_HOST}/api/auth/login'
auth_payload = {'username': USERNAME, 'password': PASSWORD}
auth_response = requests.post(auth_url, json=auth_payload)
auth_response.raise_for_status()
jwt_token = auth_response.json()['token']

In [22]:
# Load environment variables
load_dotenv()
openai_api_key = os.getenv('OPENAI_PROJECT_API_KEY')

In [16]:
# Load the device metadata
import json 
with open("farm_model_small.json", "r") as file:
    data = json.load(file)

In [13]:
# farm_data['farm']['fields'][0]

In [17]:
def get_farm_details(a: int) -> str:
    """Return Farm field details as a JSON string

    Args:
        a: Field Index
    """
    try:
        field_data = data['farm']['fields'][a]
        return json.dumps(field_data)
    except IndexError:
        return json.dumps({"error": f"Field index {a} is out of bounds."})
    except KeyError:
        return json.dumps({"error": "The 'farm' or 'fields' key was not found in the data."})
    except Exception as e:
        return json.dumps({"error": f"An error occurred: {e}"})

In [18]:
field_index = 0
farm_details_json = get_farm_details(field_index)
print(farm_details_json)

{"F001": {"name": "North Field", "crop": "Maize", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0100"], "field_air_humidity": ["HUM-0100"], "soil_conductivity": ["COND-0100"], "moisture_content": ["MOIST-0100"], "plant_health": ["CAM-0100"]}, "actuator_list": {"pumps": ["PUMP-0100", "PUMP-0101"], "water_valves": ["WV-0100"], "fertilizer_dispensers": ["FD-0100"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0100", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_humidity": [{"sensor_id": "HUM-0100", "gps": {"lat": 35.6802, "long": -98.1202}, "status": "transmitting", "unit": "grams/cubic meter"}], "soil_conductivity": [{"sensor_id": "COND-0100", "gps": {"lat": 35.6803, "long": -98.1203}, "status": "transmitting", "u

In [19]:
field_index = 6# Assuming there are fewer than 6 fields
farm_details_json = get_farm_details(field_index)
print(farm_details_json)

{"F007": {"name": "West Field", "crop": "Coffee", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0700"], "field_air_humidity": ["HUM-0700"], "soil_conductivity": ["COND-0700"], "moisture_content": ["MOIST-0700"], "plant_health": ["CAM-0700"]}, "actuator_list": {"pumps": ["PUMP-0700", "PUMP-0701"], "water_valves": ["WV-0700"], "fertilizer_dispensers": ["FD-0700"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0700", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_humidity": [{"sensor_id": "HUM-0700", "gps": {"lat": 35.6802, "long": -98.1202}, "status": "transmitting", "unit": "grams/cubic meter"}], "soil_conductivity": [{"sensor_id": "COND-0700", "gps": {"lat": 35.6803, "long": -98.1203}, "status": "transmitting", "u

In [21]:
# create tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_farm_details",
            "description": "Return details about a specific farm field as a JSON object.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "integer",
                        "description": "The index of the field to retrieve details for (0-based).",
                    },
                },
                "required": ["a"],
            },
        },
    }
]

In [45]:
import openai 
# client = openai.OpenAI(api_key=openai_api_key)
client = openai.OpenAI(api_key=openai_api_key)
llm_model = "gpt-3.5-turbo"
# client = ChatOpenAI(api_key=openai_api_key, temperature = 0.0, model=llm_model)
# client = openai.OpenAI(api_key=openai_api_key,model=llm_model)

In [59]:
def run_conversation(query):
    """Runs a conversation with the LLM that can call the get_farm_details tool."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant that identifies relevant devices (sensor_list) mentioned in the user's request and returns them as a JSON array of their IDs e.g. If no specific devices are mentioned, return an empty JSON array."
        "F001 is North Field,"
        "F002 is Northeast Field"
        "F003 is East Field"
        "F004 is Southeast Field"
        "F005 is South Field"
        "F006 is Southwest Field"
        "F007 is West Field"
        "F008 is Northwest Field"
        "F009 is Central Field"
        "Note: if the right Field is not provided return an empty json"
        },
     
        {"role": "user", "content": query}
    ]

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        tools=tools,
        tool_choice="auto", 
    )

    response_message = response.choices[0].message

    if response_message.tool_calls:
        print("LLM initiated a tool call:")
        print(response_message.tool_calls)

        tool_call = response_message.tool_calls[0]
        function_name = tool_call.function.name
        function_to_call = globals()[function_name]
        function_args = json.loads(tool_call.function.arguments)
        function_response = function_to_call(**function_args)

        print(f"Calling function '{function_name}' with arguments: {function_args}")
        print(f"Function returned: {function_response}")

        messages.append(response_message)
        messages.append(
            {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response,
            }
        )
        second_response = client.chat.completions.create(
            # model="gpt-3.5-turbo-0613",
            model="gpt-3.5-turbo",
            messages=messages,
        )
        
        return second_response.choices[0].message.content
    else:
        # If no tool call, the LLM might be directly answering based on the system prompt
        try:
            # Attempt to parse the response as JSON (assuming it followed the system prompt)
            return json.loads(response_message.content)
        except (json.JSONDecodeError, TypeError):
            # If it's not valid JSON, return the raw content
            return response_message.content

In [60]:
user_query_farm = "Tell me the details of the sensors in the south  field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_n6KcWim9g0GYRMJaaIJBVnAF', function=Function(arguments='{"a":4}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 4}
Function returned: {"F005": {"name": "South Field", "crop": "soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0500"], "field_air_humidity": ["HUM-0500"], "soil_conductivity": ["COND-0500"], "moisture_content": ["MOIST-0500"], "plant_health": ["CAM-0500"]}, "actuator_list": {"pumps": ["PUMP-0500", "PUMP-0501"], "water_valves": ["WV-0500"], "fertilizer_dispensers": ["FD-0500"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0500", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air

In [61]:
user_query_farm = "Tell me the details of the sensors in the east  field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_4KyMNYu1qd9SJAUCxqZaWeLn', function=Function(arguments='{"a":2}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 2}
Function returned: {"F003": {"name": "East Field", "crop": "Sorghum", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0300"], "field_air_humidity": ["HUM-0300"], "soil_conductivity": ["COND-0300"], "moisture_content": ["MOIST-0300"], "plant_health": ["CAM-0300"]}, "actuator_list": {"pumps": ["PUMP-0300", "PUMP-0301"], "water_valves": ["WV-0300"], "fertilizer_dispensers": ["FD-0300"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0300", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_

In [62]:
user_query_farm = "Tell me the details of the sensors in the give South field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_eCpslVjc8L3rgcpZMy9riGar', function=Function(arguments='{"a":4}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 4}
Function returned: {"F005": {"name": "South Field", "crop": "soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0500"], "field_air_humidity": ["HUM-0500"], "soil_conductivity": ["COND-0500"], "moisture_content": ["MOIST-0500"], "plant_health": ["CAM-0500"]}, "actuator_list": {"pumps": ["PUMP-0500", "PUMP-0501"], "water_valves": ["WV-0500"], "fertilizer_dispensers": ["FD-0500"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0500", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air

In [63]:
user_query_farm = "Tell me the details of the temperature sensor in the give South field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_Ybc3nIYNUGpV6RjFJtQDwhA8', function=Function(arguments='{"a":5}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 5}
Function returned: {"F006": {"name": "Southwest Field", "crop": "Soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0600"], "field_air_humidity": ["HUM-0600"], "soil_conductivity": ["COND-0600"], "moisture_content": ["MOIST-0600"], "plant_health": ["CAM-0600"]}, "actuator_list": {"pumps": ["PUMP-0600", "PUMP-0601"], "water_valves": ["WV-0600"], "fertilizer_dispensers": ["FD-0600"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0600", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field

In [70]:
user_query_farm = "Tell me the details of the temperature sensor in the give South-South field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_yUfz69meYoPqY9JnjnEE6Cqj', function=Function(arguments='{"a":5}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 5}
Function returned: {"F006": {"name": "Southwest Field", "crop": "Soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0600"], "field_air_humidity": ["HUM-0600"], "soil_conductivity": ["COND-0600"], "moisture_content": ["MOIST-0600"], "plant_health": ["CAM-0600"]}, "actuator_list": {"pumps": ["PUMP-0600", "PUMP-0601"], "water_valves": ["WV-0600"], "fertilizer_dispensers": ["FD-0600"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0600", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field

In [74]:
sensor_ids = json.loads(response_farm)["sensor_list"]
sensor_ids

['TEMP-0600']

In [75]:
# 2. Get device ID by name
def get_device_id_by_name(device_name, token):
    headers = {
        "Content-Type": "application/json",
        "X-Authorization": f"Bearer {token}"
    }
    url = f"{THINGSBOARD_URL}/api/tenant/devices?deviceName={device_name}"
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    device = response.json()
    return device['id']['id'] if device else None

In [76]:
def get_device_keys(jwt_token, device_id):
    url = f"{THINGSBOARD_URL}/api/plugins/telemetry/DEVICE/{device_id}/keys/timeseries"
    headers = {
        "X-Authorization": f"Bearer {jwt_token}"
    }

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()  # Returns a list of key names
    else:
        return {
            "error": f"Failed to fetch keys: {response.status_code}",
            "details": response.text
        }

In [77]:
# change the device name 
device_name = sensor_ids[0] #"HUM-0100"
sensor_id = get_device_id_by_name(device_name, jwt_token)
device_id = sensor_id
sensor_id

'e8340770-1221-11f0-a236-5f0808fc8cdf'

In [81]:
device_key = get_device_keys(jwt_token, sensor_id)
device_key = device_key[-1] # ['deviceId', 'unit', 'relative_humidity']
device_key 

'temp'

# Get data for the last 24 hours

In [83]:
# Last 24 hours timestamps
end_ts = int(time.time() * 1000)  
start_ts = end_ts - (340 * 60 * 60 * 1000)  # 24 hours ago

# keys = "temperature"  
keys = device_key


def get_historical_data(jwt_token, device_id, start_ts, end_ts, keys):
    url = f"{THINGSBOARD_URL}/api/plugins/telemetry/DEVICE/{device_id}/values/timeseries"
    params = {"keys": keys, "startTs": start_ts, "endTs": end_ts, "limit": 100}
    headers = {"X-Authorization": f"Bearer {jwt_token}"}
    
    response = requests.get(url, headers=headers, params=params)
    return response.json() if response.status_code == 200 else {"error": "Failed to fetch telemetry data"}

response = get_historical_data(jwt_token, device_id, start_ts, end_ts, keys)
response

{'temp': [{'ts': 1743869019663, 'value': '34.13965746815725'},
  {'ts': 1743868717745, 'value': '29.485830583629255'},
  {'ts': 1743868415323, 'value': '28.932493057746914'},
  {'ts': 1743868113219, 'value': '27.308249633328508'},
  {'ts': 1743867811319, 'value': '28.163831453041382'},
  {'ts': 1743867509257, 'value': '33.414491390775765'},
  {'ts': 1743867207358, 'value': '30.972828637899823'},
  {'ts': 1743866905295, 'value': '30.75627733284099'},
  {'ts': 1743866602940, 'value': '34.28388919327721'},
  {'ts': 1743866300937, 'value': '32.47733997039399'},
  {'ts': 1743865998751, 'value': '28.460041800723495'},
  {'ts': 1743864786041, 'value': '25.01580596035995'}]}

# Debugging devices 

In [205]:
uuid_list = [
    "e799ea50-1221-11f0-a236-5f0808fc8cdf",
    "e7a97ab0-1221-11f0-a236-5f0808fc8cdf",
    "e7acaf00-1221-11f0-a236-5f0808fc8cdf",
    "e7af6e20-1221-11f0-a236-5f0808fc8cdf",
    "e7b27b60-1221-11f0-a236-5f0808fc8cdf",
    "e7b4c550-1221-11f0-a236-5f0808fc8cdf",
    "e7b67300-1221-11f0-a236-5f0808fc8cdf",
    "e7b86ed0-1221-11f0-a236-5f0808fc8cdf",
    "e7ba6aa0-1221-11f0-a236-5f0808fc8cdf",
    "e7bc1850-1221-11f0-a236-5f0808fc8cdf",
    "e7bded10-1221-11f0-a236-5f0808fc8cdf",
    "e7bfc1d0-1221-11f0-a236-5f0808fc8cdf",
    "e7c14870-1221-11f0-a236-5f0808fc8cdf",
    "e7c2f620-1221-11f0-a236-5f0808fc8cdf",
    "e7c51900-1221-11f0-a236-5f0808fc8cdf",
    "e7c6edc0-1221-11f0-a236-5f0808fc8cdf",
    "e7c89b70-1221-11f0-a236-5f0808fc8cdf",
    "e7ca7030-1221-11f0-a236-5f0808fc8cdf",
    "e7cc9310-1221-11f0-a236-5f0808fc8cdf",
    "e7cedd00-1221-11f0-a236-5f0808fc8cdf",
    "e7d0b1c0-1221-11f0-a236-5f0808fc8cdf",
    "e7d349d0-1221-11f0-a236-5f0808fc8cdf",
    "e7d1ea40-1221-11f0-a236-5f0808fc8cdf",
    "e7d65710-1221-11f0-a236-5f0808fc8cdf",
    "e7da0090-1221-11f0-a236-5f0808fc8cdf",
    "e7ddaa10-1221-11f0-a236-5f0808fc8cdf",
    "e7e1c8c0-1221-11f0-a236-5f0808fc8cdf",
    "e7e460d0-1221-11f0-a236-5f0808fc8cdf",
    "e7e5e770-1221-11f0-a236-5f0808fc8cdf",
    "e7eaa260-1221-11f0-a236-5f0808fc8cdf",
    "e7ee4be0-1221-11f0-a236-5f0808fc8cdf",
    "e7f1f560-1221-11f0-a236-5f0808fc8cdf",
    "e7f550c0-1221-11f0-a236-5f0808fc8cdf",
    "e7f8d330-1221-11f0-a236-5f0808fc8cdf",
    "e7fc2e90-1221-11f0-a236-5f0808fc8cdf",
    "e8072b10-1221-11f0-a236-5f0808fc8cdf",
    "e80be600-1221-11f0-a236-5f0808fc8cdf",
    "e80cf770-1221-11f0-a236-5f0808fc8cdf",
    "e81079e0-1221-11f0-a236-5f0808fc8cdf",
    "e813ae30-1221-11f0-a236-5f0808fc8cdf",
    "e8164640-1221-11f0-a236-5f0808fc8cdf",
    "e819efc0-1221-11f0-a236-5f0808fc8cdf",
    "e81eaab0-1221-11f0-a236-5f0808fc8cdf",
    "e8249e20-1221-11f0-a236-5f0808fc8cdf",
    "e828bcd0-1221-11f0-a236-5f0808fc8cdf",
    "e7d4a960-1221-11f0-a236-5f0808fc8cdf",
    "e7d78f90-1221-11f0-a236-5f0808fc8cdf",
    "e7d8c810-1221-11f0-a236-5f0808fc8cdf",
    "e7db3910-1221-11f0-a236-5f0808fc8cdf",
    "e7dc7190-1221-11f0-a236-5f0808fc8cdf",
    "e7e06930-1221-11f0-a236-5f0808fc8cdf",
    "e7e76e10-1221-11f0-a236-5f0808fc8cdf",
    "e7ebdae0-1221-11f0-a236-5f0808fc8cdf",
    "e7ef8460-1221-11f0-a236-5f0808fc8cdf",
    "e7f306d0-1221-11f0-a236-5f0808fc8cdf",
    "e7f41840-1221-11f0-a236-5f0808fc8cdf",
    "e7f66230-1221-11f0-a236-5f0808fc8cdf",
    "e7f79ab0-1221-11f0-a236-5f0808fc8cdf",
    "e7f9e4a0-1221-11f0-a236-5f0808fc8cdf",
    "e7fb4430-1221-11f0-a236-5f0808fc8cdf",
    "e7fd6710-1221-11f0-a236-5f0808fc8cdf",
    "e7fe7880-1221-11f0-a236-5f0808fc8cdf",
    "e7ff89f0-1221-11f0-a236-5f0808fc8cdf",
    "e8009b60-1221-11f0-a236-5f0808fc8cdf",
    "e801acd0-1221-11f0-a236-5f0808fc8cdf",
    "e802be40-1221-11f0-a236-5f0808fc8cdf",
    "e803cfb0-1221-11f0-a236-5f0808fc8cdf",
    "e8050830-1221-11f0-a236-5f0808fc8cdf",
    "e80619a0-1221-11f0-a236-5f0808fc8cdf",
    "e7df09a0-1221-11f0-a236-5f0808fc8cdf",
    "e7e30140-1221-11f0-a236-5f0808fc8cdf",
    "e8088aa0-1221-11f0-a236-5f0808fc8cdf",
    "e80ef340-1221-11f0-a236-5f0808fc8cdf",
    "e81275b0-1221-11f0-a236-5f0808fc8cdf",
    "e8150dc0-1221-11f0-a236-5f0808fc8cdf",
    "e8205860-1221-11f0-a236-5f0808fc8cdf",
    "e82624c0-1221-11f0-a236-5f0808fc8cdf",
    "e829a730-1221-11f0-a236-5f0808fc8cdf",
    "e82e8930-1221-11f0-a236-5f0808fc8cdf",
    "e8340770-1221-11f0-a236-5f0808fc8cdf",
    "e8384d30-1221-11f0-a236-5f0808fc8cdf",
    "e8395ea0-1221-11f0-a236-5f0808fc8cdf",
    "e83b3360-1221-11f0-a236-5f0808fc8cdf",
    "e8419c00-1221-11f0-a236-5f0808fc8cdf",
    "e84370c0-1221-11f0-a236-5f0808fc8cdf",
    "e7e91bc0-1221-11f0-a236-5f0808fc8cdf",
    "e7ed3a70-1221-11f0-a236-5f0808fc8cdf",
    "e7f0bce0-1221-11f0-a236-5f0808fc8cdf",
    "e8116440-1221-11f0-a236-5f0808fc8cdf",
    "e81757b0-1221-11f0-a236-5f0808fc8cdf",
    "e8190560-1221-11f0-a236-5f0808fc8cdf",
    "e81b2840-1221-11f0-a236-5f0808fc8cdf",
    "e81d7230-1221-11f0-a236-5f0808fc8cdf",
    "e82190e0-1221-11f0-a236-5f0808fc8cdf",
    "e8227b40-1221-11f0-a236-5f0808fc8cdf",
    "e8238cb0-1221-11f0-a236-5f0808fc8cdf",
    "e8278450-1221-11f0-a236-5f0808fc8cdf",
    "e82a9190-1221-11f0-a236-5f0808fc8cdf",
    "e82c8d60-1221-11f0-a236-5f0808fc8cdf",
    "e82fc1b0-1221-11f0-a236-5f0808fc8cdf",
    "e831e490-1221-11f0-a236-5f0808fc8cdf",
    "e83518e0-1221-11f0-a236-5f0808fc8cdf",
    "e83762d0-1221-11f0-a236-5f0808fc8cdf",
    "e83a7010-1221-11f0-a236-5f0808fc8cdf",
    "e83c6be0-1221-11f0-a236-5f0808fc8cdf",
    "e83d7d50-1221-11f0-a236-5f0808fc8cdf",
    "e83eb5d0-1221-11f0-a236-5f0808fc8cdf",
    "e83fee50-1221-11f0-a236-5f0808fc8cdf",
    "e8448230-1221-11f0-a236-5f0808fc8cdf",
    "e82b7bf0-1221-11f0-a236-5f0808fc8cdf",
    "e82d9ed0-1221-11f0-a236-5f0808fc8cdf",
    "e830d320-1221-11f0-a236-5f0808fc8cdf",
    "e832cef0-1221-11f0-a236-5f0808fc8cdf",
    "e8362a50-1221-11f0-a236-5f0808fc8cdf",
    "d7f871e0-19f9-11f0-a236-5f0808fc8cdf"
]

In [ ]:
for device_id in uuid_list: 
    device_key = get_device_keys(jwt_token, device_id)
    print(device_key, device_id)